# SME Liquidity GRPO Training on Colab

This notebook follows the same SME liquidity environment.

| Component | Detail |
|-----------|--------|
| Environment | In-process SME liquidity environment from this repo |
| Training | Simple GRPO notebook wrapper |
| Model | `Qwen/Qwen3-0.6B` + LoRA |
| Framework | HF TRL GRPO using the canonical repo training path |

## 1. Install Dependencies

Install the RL dependencies needed for the canonical liquidity trainer.

Important: after this cell finishes, restart the Kaggle runtime before continuing. The cell intentionally stops with a restart reminder so NumPy / vLLM native extensions are loaded into a clean kernel.

In [ ]:
%pip install -q -U pip
%pip uninstall -y -q transformers trl vllm peft numpy scipy scikit-learn || true
%pip install -q --no-cache-dir --force-reinstall "numpy>=1.24.0,<2.0.0" "scipy>=1.10.0,<1.14.0" "scikit-learn>=1.3.0,<1.6.0"
%pip install -q --no-cache-dir --force-reinstall "trl[vllm]==0.29.0" "vllm>=0.11.0,<0.13.0" "transformers>=4.56.0,<4.57.0" "huggingface_hub>=0.34.0,<1.0" "accelerate>=1.0.0" "peft>=0.17.0,<0.20.0" bitsandbytes sentencepiece matplotlib datasets
%pip install -q --no-cache-dir --force-reinstall "fastapi>=0.104.0" "uvicorn>=0.23.0" "pydantic>=2.0.0" "httpx>=0.24.0" "websockets>=11.0,<15.0" "python-multipart>=0.0.7" "openai>=1.3.0" "python-dotenv>=1.0.1" "pyyaml>=6.0.1" "altair>=5.0.0" "pandas>=2.0.0,<2.2.2" "gymnasium>=0.28.0" "openenv-core>=0.2.3" "gradio>=4.40.0" "pyarrow>=14.0.1,<15.0.0"

from pathlib import Path

REPO_URL = "https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o.git"
REPO_DIR = Path("OpenEnv_SME_Negotiator-2.o")

if not REPO_DIR.exists():
    !git clone -q $REPO_URL

%cd $REPO_DIR
%pip install -q --no-deps -e .

import sys
from importlib.metadata import version

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {version('numpy')}")
print(f"Transformers: {version('transformers')}")
print(f"TRL         : {version('trl')}")
print(f"PEFT        : {version('peft')}")
print(f"vLLM        : {version('vllm')}")
print("Notebook bootstrap complete: training stack force-pinned, repo installed editable with --no-deps.")
raise SystemExit(
    "Restart the Kaggle runtime now, then rerun the notebook starting from Section 2. "
    "Do not continue training in this same kernel after reinstalling NumPy / vLLM dependencies."
)


## 2. Configuration

Set the model, environment task, and small profile. This notebook defaults to a T4-safe `tiny` profile.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")

# Uncomment the line below to get synchronous CUDA error messages pointing to the
# exact kernel that failed (makes training ~3× slower — for debugging only):
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

RUN_PROFILE = "tiny"
MODEL_NAME = "Qwen/Qwen3-0.6B"
TASK_NAME = "liquidity-correlation-hard"
DIFFICULTY = "hard"
TOTAL_PERIODS = 2
SEED_BASE = 1000
USE_VLLM = True
VLLM_GPU_MEMORY_UTILIZATION = 0.5
VLLM_MAX_MODEL_LENGTH = 2048
SCALE_REWARDS = "none"
STRICT_TRUSTWORTHINESS = False
EVAL_NUM_SEEDS = 4
EVAL_SEED_OFFSET = 10000
OUTPUT_DIR = Path("outputs/grpo_sme_liquidity_simple")

print(f"Profile     : {RUN_PROFILE}")
print(f"Model       : {MODEL_NAME}")
print(f"Task        : {TASK_NAME}")
print(f"Difficulty  : {DIFFICULTY}")
print(f"Periods     : {TOTAL_PERIODS}")
print(f"Seed base   : {SEED_BASE}")
print(f"Use vLLM    : {USE_VLLM}")
print(f"vLLM KV mem : {VLLM_GPU_MEMORY_UTILIZATION}")
print(f"vLLM max len: {VLLM_MAX_MODEL_LENGTH}")
print(f"Scale reward: {SCALE_REWARDS}")
print(f"Strict trust: {STRICT_TRUSTWORTHINESS}")
print(f"Eval seeds  : {EVAL_NUM_SEEDS} @ offset {EVAL_SEED_OFFSET}")
print(f"Output dir  : {OUTPUT_DIR}")
print("Note        : 'tiny' is a smoke-sized run and will produce a sparse reward curve.")
print("Recommendation: switch to profile='small' or raise num_samples/max_steps for a more meaningful curve.")
print("Backend     : the canonical path is environment-native, with trustworthiness checks and held-out evaluation in the saved manifest.")


## 3. Import Training Utilities from Package

All notebook logic comes from the repo package. The notebook stays thin and reuses the same Python training functions as the CLI script.

In [ ]:
import inspect
import rl.train_grpo_liquidity as liquidity_module

try:
    from rl.train_grpo_liquidity import (
        build_canonical_training_args,
        build_run_plan,
        build_training_session,
        make_training_args,
        plot_rewards,
        run_training,
        smoke_test_environment,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Could not import rl.train_grpo_liquidity. Run the install/bootstrap cell first so the notebook checks out "
        "the correct repo branch and installs the package in editable mode."
    ) from exc

print(f"Using rl.train_grpo_liquidity from: {Path(inspect.getfile(liquidity_module)).resolve()}")

NOTEBOOK_ARGS = make_training_args(
    profile=RUN_PROFILE,
    model_name=MODEL_NAME,
    task_name=TASK_NAME,
    difficulty=DIFFICULTY,
    total_periods=TOTAL_PERIODS,
    seed_base=SEED_BASE,
    output_dir=str(OUTPUT_DIR),
    use_vllm=USE_VLLM,
    vllm_gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    vllm_max_model_length=VLLM_MAX_MODEL_LENGTH,
    scale_rewards=SCALE_REWARDS,
    strict_trustworthiness=STRICT_TRUSTWORTHINESS,
    eval_num_seeds=EVAL_NUM_SEEDS,
    eval_seed_offset=EVAL_SEED_OFFSET,
)
CANONICAL_ARGS = build_canonical_training_args(NOTEBOOK_ARGS)
RUN_PLAN = build_run_plan(NOTEBOOK_ARGS, CANONICAL_ARGS)

RUN_PLAN


## 4. Smoke Test

Run a cheap reset-only smoke test first so the environment wiring is checked before training.

In [ ]:
smoke = smoke_test_environment(CANONICAL_ARGS)
print("Smoke test passed. Environment is ready for training.")
smoke


## 5. Train!

Launch the simple canonical GRPO training run. This uses the same Python entrypoint as `python -m rl.train_grpo_liquidity`.

In [ ]:
print("Starting GRPO training...")
print(f"  Profile     : {RUN_PROFILE}")
print(f"  Model       : {MODEL_NAME}")
print(f"  Task        : {TASK_NAME}")
print(f"  Difficulty  : {DIFFICULTY}")
print(f"  Output dir  : {OUTPUT_DIR}")
print()

print(f"Resolved canonical output dir: {RUN_PLAN['paths']['output_dir']}")

manifest = run_training(NOTEBOOK_ARGS)
training_artifacts = manifest["training"]
episode_reward_log_path = training_artifacts.get("episode_reward_log_path", training_artifacts["reward_log_path"])
trainer_reward_log_path = training_artifacts.get("trainer_reward_log_path", training_artifacts["reward_log_path"])
print(f"Checkpoint path: {training_artifacts['checkpoint_path']}")
print(f"Episode reward log path: {episode_reward_log_path}")
print(f"Trainer reward log path: {trainer_reward_log_path}")
print(f"Reward curve path: {training_artifacts['reward_curve_path']}")
print(f"Training trustworthy: {training_artifacts['training_trustworthy']}")
print(f"Trust failures: {training_artifacts['trust_failures']}")
print(f"Exposed tool names: {training_artifacts['exposed_tool_names']}")
print(f"Median reward std: {training_artifacts['median_reward_std']}")
print(f"Median unique completions: {training_artifacts['median_unique_completion_count']}")
print(f"Median identical terminal fraction: {training_artifacts['median_identical_terminal_fraction']}")
print(f"Held-out eval passed: {manifest['eval_summary']['trained_beats_base_without_extra_defaults']}")
manifest["training"]


## 6. Reward Curves

Visualize training progress using `plot_rewards` from the package.

In [ ]:
from pathlib import Path

episode_reward_log_path = Path(manifest["training"].get("episode_reward_log_path", manifest["training"]["reward_log_path"]))
output_dir = Path(manifest["plan"]["paths"]["output_dir"])

# Use plot_rewards from the package
plot_rewards(episode_reward_log_path, output_dir / "reward_curve.png")

# Also display inline
import matplotlib.pyplot as plt
from matplotlib.image import imread

img = imread(str(output_dir / "reward_curve.png"))
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(img)
ax.axis("off")
plt.show()

print(f"Episode reward log saved to {episode_reward_log_path}")
print(f"Reward curve saved to {output_dir / 'reward_curve.png'}")


## 7. Trust and Held-Out Evaluation

Review the trustworthiness gate and the saved held-out evaluation summary before trusting the checkpoint.

In [ ]:
eval_summary = manifest["eval_summary"]
print(f"Overall held-out eval passed: {eval_summary['trained_beats_base_without_extra_defaults']}")
for task_name, payload in eval_summary["tasks"].items():
    print(f"{task_name}: passed={payload['trained_beats_base_without_extra_defaults']}")
    print(f"  Eval summary path: {payload['eval_summary_path']}")
    print(f"  Policy comparison path: {payload['policy_comparison_path']}")

eval_summary


## 8. Exported Artifacts

The simple script writes a run manifest plus the main training artifacts.

In [ ]:
training_artifacts = manifest["training"]
print(f"Checkpoint path: {training_artifacts['checkpoint_path']}")
print(f"Episode reward log path: {training_artifacts.get('episode_reward_log_path', training_artifacts['reward_log_path'])}")
print(f"Trainer reward log path: {training_artifacts.get('trainer_reward_log_path', training_artifacts['reward_log_path'])}")
print(f"Reward curve path: {training_artifacts['reward_curve_path']}")
print(f"Training dashboard path: {training_artifacts['training_dashboard_path']}")
print(f"Run manifest path: {manifest['manifest_path']}")

manifest
